Let's start by loading up a real subset of Wikipedia data I've placed in S3 for you.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Seems there are no column names in our source data, so we'll have to add them ourselves:

In [2]:
# Step 1: Define documents
docA = "The cat sat on my face"
docB = "The dog sat on my bed"
documents = [docA, docB]

Next we need to "clean" our data. We know TF/IDF can't handle null documents, so first let's check for that.

In [3]:
# Step 2: Text Preprocessing Function
def preprocess_text(docs):
    # Convert to lowercase, remove punctuation, and tokenize words
    processed_docs = [doc.lower().split() for doc in docs]
    return processed_docs

processed_docs = preprocess_text(documents)

Looks like there is one null document. As there is only one and it's clearly corrupt when we look into it, we can just remove it and call it a day.

In [4]:
# Step 3: Compute TF-IDF using sklearn for better efficiency
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)


TF/IDF wants numbers, not words. So now we need to pre-process our data before we can run any fun algorithms on it. We'll first tokenize the articles to split them up into words, and store them in a sparse vector that is now a numeric representation of the words in each article.

In [5]:
# Step 4: Create DataFrame of TF-IDF values
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())


In [6]:
# Display TF-IDF matrix
print("TF-IDF Matrix:")
display(tfidf_df)  # Using display() for better output in Jupyter Notebook

TF-IDF Matrix:


,bed,cat,dog,face,my,on,sat,the
0,0.000000,0.498446,0.000000,0.498446,0.354649,0.354649,0.354649,0.354649
1,0.498446,0.000000,0.498446,0.000000,0.354649,0.354649,0.354649,0.354649


That hashing operation basically computed term frequencies for us by storing how often each hashed word occured in each article. So we have TF, but we want TF/IDF scores for every term in every document. We'll store these final scores in a new column called "features", which is a sparse vector containing TF/IDF scores for each feature.

In [7]:
# Step 5: Cosine Similarity Computation for Document Similarity
cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim_df = pd.DataFrame(cosine_sim, index=["DocA", "DocB"], columns=["DocA", "DocB"])


In [8]:
print("\nCosine Similarity between documents:")
print(cosine_sim_df)



Cosine Similarity between documents:
          DocA      DocB
DocA  1.000000  0.503103
DocB  0.503103  1.000000


So let's use this to do a search for the term "Gettysburg". Again, we need numbers, not words, so the first task is to get the hash value for "Gettysburg"

In [ ]:
# Optional: Visualization of the TF-IDF Matrix and Cosine Similarity Matrix
import matplotlib.pyplot as plt
import seaborn as sns


OK, we have the magic number that represents "Gettysburg". Now we can add another column - we'll call it "score" - that just extracts the TF/IDF value for Gettysburg for each document.

In [ ]:
# TF-IDF Heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(tfidf_df, annot=True, cmap="YlGnBu")
plt.title("TF-IDF Matrix Heatmap")
plt.show()



Now all we have to do is sort our articles by score, and we'll have the most relevant articles for Gettysburg!

In [ ]:
# Cosine Similarity Heatmap
plt.figure(figsize=(5, 3))
sns.heatmap(cosine_sim_df, annot=True, cmap="coolwarm")
plt.title("Cosine Similarity between Documents")
plt.show()